In [3]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os

os.listdir("./outputs")

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, median_absolute_error, r2_score
from scipy.stats import pearsonr
import polars as pl
import numpy as np
from tqdm import tqdm
from datetime import datetime
import os

train_splits = {
    "full" : pl.datetime(2023, 3, 1, 0, 0, 0),
    "last_12m" : pl.datetime(2023, 6, 1, 0, 0, 0),
    "last_9m" : pl.datetime(2023, 9, 1, 0, 0, 0),
    "last_3m" : pl.datetime(2023, 12, 1, 0, 0, 0),
    "last_1m": pl.datetime(2024, 2, 1, 0, 0, 0),
}

PATHS = {
    "TRAIN_PATH" :"./kaggle/kaggle/input/drw-crypto-market-prediction/train.parquet",
    "TEST_PATH" : "./kaggle/kaggle/input/drw-crypto-market-prediction/test.parquet",
    "SUBMISSION_PATH" : "/kaggle/input/drw-crypto-market-prediction/sample_submission.csv",
}

train_data = pl.read_parquet(PATHS["TRAIN_PATH"]).sort("timestamp", descending = False)
print(train_data)

shape: (525_887, 897)
┌─────────┬─────────┬─────────┬──────────┬───┬──────────┬──────────┬──────────┬──────────────┐
│ bid_qty ┆ ask_qty ┆ buy_qty ┆ sell_qty ┆ … ┆ X889     ┆ X890     ┆ label    ┆ timestamp    │
│ ---     ┆ ---     ┆ ---     ┆ ---      ┆   ┆ ---      ┆ ---      ┆ ---      ┆ ---          │
│ f64     ┆ f64     ┆ f64     ┆ f64      ┆   ┆ f64      ┆ f64      ┆ f64      ┆ datetime[ns] │
╞═════════╪═════════╪═════════╪══════════╪═══╪══════════╪══════════╪══════════╪══════════════╡
│ 15.283  ┆ 8.425   ┆ 176.405 ┆ 44.984   ┆ … ┆ 0.159183 ┆ 0.530636 ┆ 0.562539 ┆ 2023-03-01   │
│         ┆         ┆         ┆          ┆   ┆          ┆          ┆          ┆ 00:00:00     │
│ 38.59   ┆ 2.336   ┆ 525.846 ┆ 321.95   ┆ … ┆ 0.158963 ┆ 0.530269 ┆ 0.533686 ┆ 2023-03-01   │
│         ┆         ┆         ┆          ┆   ┆          ┆          ┆          ┆ 00:01:00     │
│ 0.442   ┆ 60.25   ┆ 159.227 ┆ 136.369  ┆ … ┆ 0.158744 ┆ 0.529901 ┆ 0.546505 ┆ 2023-03-01   │
│         ┆         ┆       

# Data

In [8]:
from numpy import ndarray
from typing import Any
from sklearn.decomposition import IncrementalPCA
from sklearn.feature_selection import f_regression, SelectKBest
import polars as pl

class FeatureEngineeringPipeline:
    def __init__(
        self,
        data: pl.DataFrame,
        y_col: str = "label",
        drop_columns: list[str] = None,
        config: dict = None
    ):
        """
        config = {
            "model_agnostic": {
                "preprocessing": {…},
                "transformation": {"poly_degree": 2, …},
                "filter": {"method": "SelectPercentile", "percentile": 10},
                "extraction": {"method": "PCA", "n_components": 5}
            },
            "model_based": {
                "embedded": {"alpha": 1.0},
                "wrapper": {"n_features_to_select": 10},
                "perm_importance": {"n_repeats": 5},
                "stability": {"n_bootstrap": 50}
            },
            "aggregation": {"strategy": "rank_sum", "weights": {...}}
        }
        """
        self.data = data.lazy()
        self.y_col = y_col
        self.y = data.select(pl.col(y_col)).to_numpy()
        self.drop_columns = drop_columns or []
        self.config = config or {}


    def run_feature_engineering_pipeline(self):
        """
        A comprehensive quantitative feature engineering pipeline.
        """
        pass

    def run_model_agnostic_selection(self, lazy_df: pl.LazyFrame) -> tuple[pl.LazyFrame, list[str]]:
        """
        Pre-model evaluation:
        1. Data Preprocessing 
            Handle missing, infinite or outlier values to ensure numerical stability (e.g. drop or impute NaNs, clip infinities or extreme quantiles).
        2. Scaling/Normalization
            Standardize variances (z-score) or map to fixed ranges (min-max, log-scaling) so all features live on comparable numerical scales.
        3. Add Features / Categorical Encoding
            Convert categoricals (one-hot, ordinal), date/times (e.g. cyclic encodings), and build simple interactions or aggregated statistics (ratios, deltas).
        4. Feature Transformation / Generation
            - Transformation: apply deterministic mappings column-wise—e.g. PolynomialFeatures, FunctionTransformer, PowerTransformer, QuantileTransformer, or hash-based projections (FeatureHasher).
            - Generation: assemble parallel or heterogeneous pipelines via FeatureUnion or ColumnTransformer to combine, conditionally apply or concatenate multiple transforms.
        5. Unsupervised Extraction
            - Extraction: reduce dimensionality by projecting into latent subspaces (PCA, TruncatedSVD, ICA, NMF) using only input covariances or non-negativity constraints.
        6. Filter methods 
            - Filter selection: remove low-value inputs via univariate criteria—variance thresholds, correlation filters, or statistical tests with error-rate control (e.g. SelectKBest, SelectPercentile, SelectFwe, SelectFdr).
        """
        all_drop_cols = []
        # list your step‐functions in order
        stages = [
            self._data_preprocessing_1, # 1. Data Preprocessing
            self._scaling_or_norm_2, # 2. Scaling/Normalization
            self._addfeaturesandcategorical_3, # 3. Add Features / Categorical Encoding
            self._featuretransformandgenreation_4,  # 4. Feature Transformation / Generation
            self._unsupervisedextraction_5, # 5. Unsupervised Extraction
            self._filtermethods_6, # 6. Filter methods
        ]
        for fn in stages:
            lazy_df, drop_list = fn(lazy_df)
            all_drop_cols.extend(drop_list)
        return lazy_df, all_drop_cols
    
    def run_model_based_selection(self):
        """
        Model evaluation:
        1. Embedded Methods - Lasso, Importance
        2. Wrapper - RFE, sequential
        3. Model agnostic importance - SHAP, Permutation Importance
        4. Stability Selection - 
        """
        pass

    def _data_preprocessing_1(self, lazy_df: pl.LazyFrame) -> tuple[pl.LazyFrame, list[str]]:
        # Collect the lazy frame to get column information
        df_collected = lazy_df.collect()
        cols = df_collected.drop("timestamp").columns
        
        # Check for infinite values
        inf_flags = df_collected.select([pl.col(c).is_infinite().any().alias(c) for c in cols]).row(0)
        inf_cols = [c for c, flag in zip(cols, inf_flags) if flag]

        # Check for NaN values
        nan_flags = df_collected.select([pl.col(c).is_null().any().alias(c) for c in cols]).row(0)
        nan_cols = [c for c, flag in zip(cols, nan_flags) if flag]

        # Check for zero standard deviation
        numeric_cols = [c for c, dt in zip(df_collected.columns, df_collected.dtypes) if dt.is_numeric()]
        std_flags = (
            df_collected
            .select([pl.col(c).std().alias(c) for c in numeric_cols])
            .row(0)
        )
        zerostd_cols = [c for c, std in zip(numeric_cols, std_flags) if std == 0 or std is None]

        drop_cols = inf_cols + nan_cols + zerostd_cols + self.drop_columns
        return lazy_df.drop(drop_cols), drop_cols
    
    def _scaling_or_norm_2(self, lazy_df: pl.LazyFrame) -> tuple[pl.LazyFrame, list[str]]:
        # Placeholder - no scaling implemented yet
        return lazy_df, []
    
    def _addfeaturesandcategorical_3(self, lazy_df: pl.LazyFrame) -> tuple[pl.LazyFrame, list[str]]:
        lazy_df = lazy_df.with_columns([
            (pl.col("bid_qty") / pl.col("ask_qty")).alias("bidask_ratio"),
            pl.when(pl.col("volume") == 0)
            .then(0)
            .otherwise(pl.col("buy_qty") / pl.col("sell_qty"))
            .alias("buysell_ratio"),
            (pl.col("bid_qty") - pl.col("ask_qty")).alias("bidask_delta"),
            (pl.col("buy_qty") - pl.col("sell_qty")).alias("buysell_delta"),
            (pl.col("buy_qty") + pl.col("sell_qty")).alias("buysell_size"),
            (pl.col("bid_qty") + pl.col("ask_qty")).alias("bidask_size"),
        ])
        drop_cols = ["bid_qty", "ask_qty", "buy_qty", "sell_qty"]
        return lazy_df.drop(drop_cols), drop_cols

    def _featuretransformandgenreation_4(self, lazy_df: pl.LazyFrame) -> tuple[pl.LazyFrame, list[str]]:
        return lazy_df, []

    def _unsupervisedextraction_5(self, lazy_df: pl.LazyFrame) -> tuple[pl.LazyFrame, list[str]]:
        # Incremental PCA for dimensionality reduction
        df = lazy_df.collect()
        ipca: IncrementalPCA = IncrementalPCA(n_components=5)
        ipca.fit(df).components_.T
        load  = pl.DataFrame(data=ipca.fit(df).components_.T, schema=df.columns)
        thresh = load.select(pl.all().abs().mean()).unpivot().select(pl.col("value")).mean().item()
        features = (
            load
            .select(pl.all().abs().max())      # one‐row DF of max(|x|) per column
            .unpivot()                            # long form with "variable" & "value"
            .filter(pl.col("value") >= thresh)  # keep only those above thresh
            .select("variable")                # just the column names
            .to_series()                       # Series of names
            .to_list()                         # Python list
        )
        # Get columns to drop (all columns except timestamp and selected features)
        all_cols = df.columns
        drop_cols = [col for col in all_cols if col not in features]
        
        return lazy_df.select(*features), drop_cols
        
    def _filtermethods_6(self, lazy_df: pl.LazyFrame) -> tuple[pl.LazyFrame, list[str]]:
        df = lazy_df.collect()
        features = df.columns
        X = lazy_df.collect().to_numpy()
        selector = SelectKBest(score_func=f_regression, k=10)
        selector.fit(X, self.y)
        mask = selector.get_support()
        selected_features = [name for name, selected in zip(features, mask) if selected]
        return lazy_df.select(selected_features), []

    def __get_config(self, key: str, default=None):
        """
        Retrieve a configuration value, with optional default.
        """
        return self.config.get(key, default)
    
fepipeline = FeatureEngineeringPipeline(train_data, y_col="label", drop_columns=[])
fepipeline

# 3 Feature Engineering

In [11]:
X = train_data.drop(["timestamp", "label"])
X

bid_qty,ask_qty,buy_qty,sell_qty,volume,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15,X16,X17,X18,X19,X20,X21,X22,X23,X24,X25,X26,X27,X28,X29,X30,X31,X32,…,X854,X855,X856,X857,X858,X859,X860,X861,X862,X863,X864,X865,X866,X867,X868,X869,X870,X871,X872,X873,X874,X875,X876,X877,X878,X879,X880,X881,X882,X883,X884,X885,X886,X887,X888,X889,X890
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
15.283,8.425,176.405,44.984,221.389,0.121263,-0.41769,0.005399,0.125948,0.058359,0.027359,0.03578,0.068219,1.034825,-0.029575,0.327805,0.485823,0.668596,0.617389,0.770037,0.857631,1.754456,0.572503,0.883229,0.58567,0.816321,0.529973,0.508244,0.448616,1.341892,1.406392,0.953631,1.183991,1.474789,0.774389,0.660586,0.269043,…,0.2734375,0.418618,-0.216525,0.200508,0.492433,-0.51249,0.541286,-0.336399,-1.027483,0.21857,0.0,1.728155,0.62414,0.0,-0.051211,0.0,0.0,0.0,0.0,0.691754,0.242124,2.096157,3.369195,0.244667,0.286611,0.722679,0.901931,1.000007,1.925423,1.847943,0.005676,0.190791,0.369691,0.37763,0.210153,0.159183,0.530636
38.59,2.336,525.846,321.95,847.796,0.302841,-0.049576,0.356667,0.481087,0.237954,0.208359,0.217057,0.249624,0.948694,-0.183488,0.150526,0.308421,0.492232,0.529787,0.682958,0.770965,1.686504,0.273357,0.591695,0.442391,0.674792,0.460741,0.439681,0.380399,1.304113,1.003783,0.776628,1.015943,1.312735,0.696895,0.584217,0.231104,…,0.273481,0.424977,-0.180112,0.213252,0.479806,-0.180527,0.450331,-0.31915,-1.024055,0.088014,0.0,1.665698,0.622775,0.0,-0.079621,0.0,0.0,0.0,0.0,0.691665,0.242091,2.46103,4.127584,0.321394,0.31246,0.746452,0.912371,1.003153,1.928569,1.849468,0.005227,0.18466,0.363642,0.374515,0.209573,0.158963,0.530269
0.442,60.25,159.227,136.369,295.596,0.167462,-0.291212,0.083138,0.206881,0.101727,0.072778,0.081564,0.114166,0.896459,-0.261779,0.044571,0.200608,0.384558,0.476229,0.629848,0.718232,1.656707,0.140156,0.457268,0.376524,0.610116,0.429751,0.409316,0.350359,1.28325,0.760801,0.670816,0.917205,1.219124,0.653355,0.541739,0.210095,…,0.273524,0.409942,-0.265966,0.191734,0.440207,-0.108209,0.420681,-0.316953,-1.024056,-0.147363,0.0,1.666893,0.621414,0.0,-0.080427,0.0,0.0,0.0,0.0,0.691674,0.242093,2.493249,4.182112,0.326701,0.314636,0.746681,0.911129,1.002502,1.928047,1.849282,0.004796,0.178719,0.357689,0.371424,0.208993,0.158744,0.529901
4.865,21.016,335.742,124.963,460.705,0.072944,-0.43659,-0.102483,0.017551,0.007149,-0.021681,-0.012936,0.019634,0.732634,-0.535845,-0.273947,-0.124959,0.056438,0.311539,0.465377,0.554022,1.663491,0.152084,0.468778,0.383696,0.618529,0.435326,0.415523,0.356895,1.319538,0.955549,0.789646,1.044941,1.353001,0.72392,0.613462,0.246212,…,0.273568,0.400075,-0.322244,0.183687,0.404295,-0.169373,0.386584,-0.314775,-1.024058,-0.09459,0.0,1.735322,0.620057,0.0,-0.094702,0.0,0.0,0.0,0.0,0.69121,0.24193,2.525526,4.292975,0.350791,0.32357,0.753829,0.913363,1.002985,1.928621,1.849608,0.004398,0.172967,0.351832,0.368358,0.208416,0.158524,0.529534
27.158,3.451,98.411,44.407,142.818,0.17382,-0.213489,0.096067,0.215709,0.107133,0.078976,0.087818,0.120426,0.763537,-0.430945,-0.205298,-0.062118,0.117266,0.341493,0.495591,0.584519,1.668419,0.156177,0.472732,0.3871,0.623192,0.439034,0.419868,0.361572,1.324595,0.90546,0.78375,1.047708,1.36188,0.732001,0.622712,0.251095,…,0.273611,0.391759,-0.369625,0.192377,0.415438,-0.198976,0.389969,-0.312628,-1.02406,0.162221,0.0,1.712096,0.618703,0.0,-0.091884,0.0,0.0,0.0,0.0,0.691207,0.241928,2.52443,4.306694,0.335599,0.31907,0.747533,0.908904,1.001286,1.927084,1.84895,0.004008,0.167391,0.346066,0.365314,0.207839,0.158304,0.529167
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
4.163,6.805,39.037,55.351,94.388,0.020155,0.0

In [12]:
filter, cols = fepipeline._filtermethods_6(X.lazy())
filter

AttributeError: 'FeatureEngineeringPipeline' object has no attribute 'y'

## Incremental PCA

In [ ]:
# from sklearn.decomposition import IncrementalPCA

# ipca: IncrementalPCA = IncrementalPCA(n_components=5)
# ipca.fit(X).components_.T
# load  = pl.DataFrame(data=ipca.fit(X).components_.T, schema=X.columns)
# thresh = load.select(pl.all().abs().mean()).unpivot().select(pl.col("value")).mean().item()
# features = (
#     load
#     .select(pl.all().abs().max())      # one‐row DF of max(|x|) per column
#     .unpivot()                            # long form with “variable” & “value”
#     .filter(pl.col("value") >= thresh)  # keep only those above thresh
#     .select("variable")                # just the column names
#     .to_series()                       # Series of names
#     .to_list()                         # Python list
# )
# len(features)

168